In [ ]:
import os, re, json
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple, Any, Optional

import numpy as np
import requests
import chromadb

try:
    from tqdm import tqdm
except Exception:
    tqdm = None


import textwrap



def pretty_print(*args):
    text = " ".join(str(arg) for arg in args)
    try:
        print(textwrap.fill(text, width=80))
    except Exception as e:
        print(text)  # fallback to normal print if text is not a string


@dataclass
class Settings:
    # Ollama
    OLLAMA_URL: str = os.getenv("OLLAMA_URL", "http://localhost:11434")
    CHAT_MODEL: str = os.getenv("CHAT_MODEL", "llama3.1:8b")
    # Prefer a common embedding model mentioned in prereqs:
    EMBED_MODEL: str = os.getenv("EMBED_MODEL", "embeddinggemma:latest")

    # Chunking
    WORDS_PER_CHUNK: int = int(os.getenv("WORDS_PER_CHUNK", "300"))
    OVERLAP_WORDS: int = int(os.getenv("OVERLAP_WORDS", "60"))

    # Retrieval
    TOPK: int = int(os.getenv("TOPK", "5"))

    # Storage
    CORPUS_DIR: str = os.getenv("CORPUS_DIR", "corpus_jupyter")
    ARTIFACTS_DIR: str = os.getenv("ARTIFACTS_DIR", "rag_artifacts")
    CHROMA_PATH: str = os.getenv("CHROMA_PATH", "./chroma_db")
    CHROMA_COLLECTION: str = os.getenv("CHROMA_COLLECTION", "gutenberg_demo")
    CHROMA_DISTANCE: str = os.getenv("CHROMA_DISTANCE", "cosine")  # hnsw space

    # Workflow toggles
    DOWNLOAD_FROM_WEB: bool = (os.getenv("DOWNLOAD_FROM_WEB", "true").lower() == "true")
    FORCE_REBUILD: bool = (os.getenv("FORCE_REBUILD", "false").lower() == "true")

    # Performance
    EMBED_BATCH_SIZE: int = int(os.getenv("EMBED_BATCH_SIZE", "64"))

S = Settings()
S